In [2]:
import requests
import pandas as pd

# Координаты Алматы и даты
url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 43.2389,
    "longitude": 76.8897,
    "start_date": "2020-02-10",
    "end_date": "2024-08-12",
    "daily": ["temperature_2m_mean", "wind_speed_10m_max"],
    "hourly": ["relative_humidity_2m", "surface_pressure"], # Добавили давление сюда
    "timezone": "Asia/Almaty"
}

response = requests.get(url, params=params)
data = response.json()

# 1. Ежедневные данные (Температура и Ветер)
weather_daily = pd.DataFrame(data["daily"])
weather_daily.columns = ['ds', 'temp', 'wind_speed']
weather_daily['ds'] = pd.to_datetime(weather_daily['ds'])

# 2. Почасовые данные (Влажность + Давление)
hourly_data = pd.DataFrame(data["hourly"])
hourly_data['time'] = pd.to_datetime(hourly_data['time'])

# Группируем оба параметра сразу
hourly_daily = hourly_data.groupby(hourly_data['time'].dt.date).agg({
    'relative_humidity_2m': 'mean',
    'surface_pressure': 'mean'
}).reset_index()

hourly_daily.columns = ['ds', 'humidity', 'pressure']
hourly_daily['ds'] = pd.to_datetime(hourly_daily['ds'])

# 3. Финальное объединение
weather_df = pd.merge(weather_daily, hourly_daily, on='ds')

# Округляем до 2 знаков для чистоты
weather_df = weather_df.round(2)

# Результат
weather_df

C:\Users\user\AppData\Local\Temp\ipykernel_10652\1620779776.py:41: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  weather_df = weather_df.round(2)


,ds,temp,wind_speed,humidity,pressure
0,2020-02-10,5.6,10.1,78.29,918.64
1,2020-02-11,-3.5,16.2,90.83,924.91
2,2020-02-12,-1.2,20.8,68.04,918.59
3,2020-02-13,-0.6,12.3,86.62,927.45
4,2020-02-14,-2.5,11.0,91.33,932.97
...,...,...,...,...,...
1641,2024-08-08,27.1,7.8,31.33,917.87
1642,2024-08-09,29.0,28.9,31.12,915.04
1643,2024-08-10,27.2,21.6,40.04,918.13
1644,2024-08-11,24.2,9.4,50.92,919.62


Wind Speed (wind_speed_10m_max)
Measurement: km/h (kilometers per hour).

Parameter	API Key	Unit	Typical Range in Almaty
Wind Speed	wind_speed_10m_max	km/h	5 – 40 km/h
Pressure	surface_pressure	hPa	900 – 935 hPa
Temperature	temperature_2m_mean	°C	-20°C to +35°C
Humidity	relative_humidity_2m	%	20% to 90%

In [ ]:
'''Parameter	API Key	Unit	Typical Range in Almaty
Wind Speed	wind_speed_10m_max	km/h	5 – 40 km/h
Pressure	surface_pressure	hPa	900 – 935 hPa
Temperature	temperature_2m_mean	°C	-20°C to +35°C
Humidity	relative_humidity_2m	%	20% to 90%'''

In [3]:
weather_df.to_csv('almaty_weather.csv', index=False,float_format='%.2f')